- Dùng tuple để lưu trữ và tra cứu trạng thái của bàn cờ
- 1 đại diện cho X, -1 đại diện cho O, 0 đại diện cho ô trống

In [1]:
class TicTacToe:
  def __init__(self):
    self.board = [0] * 9
    self.current_winner = None

  def get_state(self):
    return tuple(self.board) # chuyển mảng trạng thái thành một tuple (bất biến) dùng làm key tra cứu

  def available_moves(self):
    moves = []
    for i in range(9):
      if self.board[i] == 0:
        moves.append(i)
    return moves
  def make_move(self, square, player):
    if self.board[square] == 0:
      self.board[square] = player
      if self.check_win(player):
        self.current_winner = player
      return True
    else:
      return False

  def check_win(self, player):
    win_conditions = [[0, 1, 2], [3, 4, 5], [6, 7, 8],
                      [0, 3, 6], [1, 4, 7], [2, 5, 8],
                      [0, 4, 8], [2, 4, 6]]
    for conditions in win_conditions:
      if all(self.board[i] == player for i in conditions):
        return True
    return False

  def is_full(self):
    return 0 not in self.board


In [2]:
import random

class RLAgent:
  def __init__(self, player_symbol, learning_rate = 0.1, epsilon = 0.2):
    self.player = player_symbol
    self.learning_rate = learning_rate
    self.epsilon = epsilon
    self.V = {}
    self.state_history = []

  def choose_action(self, game):
    moves = game.available_moves()
    if random.uniform(0,1) < self.epsilon:
      return random.choice(moves)

    best_value = -float('inf')
    best_move = None
    for move in moves:
      game.board[move] = self.player
      next_state = game.get_state()
      game.board[move] = 0
      if next_state not in self.V:
        self.V[next_state] = 0.5
      if self.V[next_state] > best_value:
        best_value = self.V[next_state]
        best_move = move
    return best_move

  def update_values(self, reward):
    final_state = self.state_history[-1]
    self.V[final_state] = reward
    for i in reversed(range(len(self.state_history) - 1)):
      state = self.state_history[i]
      next_state = self.state_history[i+1]
      if state not in self.V:
        self.V[state] = 0.5
      self.V[state] = self.V[state] + self.learning_rate * (self.V[next_state] - self.V[state])
    self.state_history = []


In [3]:
# tại sao lúc nãy thì nói là không cần chờ hết ván mới đánh giá cập nhật được
# mà giờ lại phải chờ rõ thắng thua mới quay ngược lại cập nhật
# tại vì: các phần thưởng trung gian đều là 0 chỉ khi hết ván mới biết có thắng không
# để tăng giái trị của các trạng thái trước đó lên
# bản chất vẫn là cập nhật từng bước một, chứ không gán phần thưởng cho toàn bộ trạng thái như phương pháp tiến hóa

In [4]:
def train_agent(episodes = 10000):
  agent_X = RLAgent(player_symbol=1)
  for episode in range(episodes):
    game = TicTacToe()
    current_player = 1
    while not game.is_full() and game.current_winner is None:
      if current_player == 1:
        move = agent_X.choose_action(game)
        game.make_move(move, 1)
        agent_X.state_history.append(game.get_state())
      else:
        move = random.choice(game.available_moves())
        game.make_move(move, -1)
      current_player = -current_player

    if game.current_winner == 1:
      reward_X = 1
    else:
      reward_X = 0
    agent_X.update_values(reward_X)

  return agent_X

In [5]:
trained_agent = train_agent(episodes=10000)

print("Số trạng thái đã học được:")
print(len(trained_agent.V))

first_5_states = list(trained_agent.V.items())[:5]
for state, value in first_5_states:
    print(f"State: {state} => Value: {value:.4f}")

Số trạng thái đã học được:
2391
State: (1, 0, 0, 0, 0, 0, 0, 0, 0) => Value: 0.8473
State: (0, 1, 0, 0, 0, 0, 0, 0, 0) => Value: 0.7904
State: (0, 0, 1, 0, 0, 0, 0, 0, 0) => Value: 0.8332
State: (0, 0, 0, 1, 0, 0, 0, 0, 0) => Value: 0.7380
State: (0, 0, 0, 0, 1, 0, 0, 0, 0) => Value: 0.8006


In [ ]:
def test_agent_vs_random(agent, num_games=1000):
    original_epsilon = agent.epsilon
    agent.epsilon = 0
    wins = 0
    losses = 0
    draws = 0
    for _ in range(num_games):
        game = TicTacToe()
        current_player = 1 # cho X đi trước

        while not game.is_full() and game.current_winner is None:
            if current_player == agent.player:
                move = agent.choose_action(game)
                game.make_move(move, agent.player)
            else:
                move = random.choice(game.available_moves()) # O đánh ngẫu nhiên
                game.make_move(move, -1)
            current_player *= -1
        if game.current_winner == agent.player:
            wins += 1
        elif game.current_winner is None:
            draws += 1
        else:
            losses += 1

    agent.epsilon = original_epsilon

    print(f"\n--- KẾT QUẢ TEST SAU {num_games} VÁN ---")
    print(f"Thắng: {wins} ván")
    print(f"Hòa:   {draws} ván")
    print(f"Thua:  {losses} ván")
trained_agent = train_agent(episodes=10000)
test_agent_vs_random(trained_agent, 1000)


--- KẾT QUẢ TEST SAU 1000 VÁN ---
Thắng: 993 ván
Hòa:   7 ván
Thua:  0 ván



PHẦN NÀY AI GEN - KHÔNG TỰ CODE
---



In [7]:
def print_board(board):
    # Hàm phụ để in bàn cờ ra màn hình cho dễ nhìn
    symbols = {1: 'X', -1: 'O', 0: ' '}
    print("\n-------------")
    for row in range(3):
        print(f"| {symbols[board[row*3]]} | {symbols[board[row*3+1]]} | {symbols[board[row*3+2]]} |")
        print("-------------")

def play_vs_human(agent):
    print("\nCHÀO MỪNG BẠN ĐẾN VỚI TRẬN CHIẾN NGƯỜI VÀ MÁY!")
    print("Các ô được đánh số từ 0 đến 8 như sau:")
    print("0 | 1 | 2\n---------\n3 | 4 | 5\n---------\n6 | 7 | 8")

    original_epsilon = agent.epsilon
    agent.epsilon = 0 # Tắt khám phá

    game = TicTacToe()
    current_player = 1 # AI (X) đi trước

    while not game.is_full() and game.current_winner is None:
        if current_player == agent.player:
            print("\nLượt của AI (X)...")
            move = agent.choose_action(game)
            game.make_move(move, agent.player)
        else:
            print_board(game.board)
            valid_move = False
            while not valid_move:
                try:
                    move = int(input("\nĐến lượt bạn (O). Nhập ô muốn đánh (0-8): "))
                    if move in game.available_moves():
                        game.make_move(move, -1)
                        valid_move = True
                    else:
                        print("Ô này đã có người đánh hoặc không hợp lệ. Vui lòng chọn lại!")
                except ValueError:
                    print("Vui lòng nhập một số nguyên!")

        current_player *= -1

    print_board(game.board)
    if game.current_winner == agent.player:
        print("AI: 'Khè khè, loài người các ngươi còn non lắm!' (AI THẮNG)")
    elif game.current_winner is None:
        print("HÒA! Trận chiến bất phân thắng bại.")
    else:
        print("BẠN ĐÃ CHIẾN THẮNG! AI cần được train thêm.")

    agent.epsilon = original_epsilon

play_vs_human(trained_agent)


CHÀO MỪNG BẠN ĐẾN VỚI TRẬN CHIẾN NGƯỜI VÀ MÁY!
Các ô được đánh số từ 0 đến 8 như sau:
0 | 1 | 2
---------
3 | 4 | 5
---------
6 | 7 | 8

Lượt của AI (X)...

-------------
|   |   | X |
-------------
|   |   |   |
-------------
|   |   |   |
-------------

Đến lượt bạn (O). Nhập ô muốn đánh (0-8): 4

Lượt của AI (X)...

-------------
|   |   | X |
-------------
|   | O | X |
-------------
|   |   |   |
-------------

Đến lượt bạn (O). Nhập ô muốn đánh (0-8): 8

Lượt của AI (X)...

-------------
| X |   | X |
-------------
|   | O | X |
-------------
|   |   | O |
-------------

Đến lượt bạn (O). Nhập ô muốn đánh (0-8): 1

Lượt của AI (X)...

-------------
| X | O | X |
-------------
|   | O | X |
-------------
|   | X | O |
-------------

Đến lượt bạn (O). Nhập ô muốn đánh (0-8): 6

Lượt của AI (X)...

-------------
| X | O | X |
-------------
| X | O | X |
-------------
| O | X | O |
-------------
HÒA! Trận chiến bất phân thắng bại.
